<a href="https://colab.research.google.com/github/JdogLloyd1/DeepLearning5888Project/blob/main/5888_PiczakProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Imports and configuration

%pip install -q librosa

import os
import time
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

# Wall-clock timing: captured as early as possible so the final elapsed
# time covers the whole notebook from import to summary.
NOTEBOOK_START_TIME = time.time()


def _format_elapsed(seconds):
    """Format a seconds value as HH:MM:SS for log lines."""
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


SEED = 99
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#ZIP_PATH = '/content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/piczak_dataset.zip'
ZIP_PATH = '/content/drive/MyDrive/5888_Project/Dataset/piczak_dataset.zip'
EXTRACT_DIR = '/content/piczak_dataset'
OUTPUT_DIR = '/content/drive/MyDrive/5888_Project/results/Phase1_Baseline'

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_file:
        zip_file.extractall(EXTRACT_DIR)

CSV_PATH = None
AUDIO_DIR = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'esc50.csv' in files:
        CSV_PATH = os.path.join(root, 'esc50.csv')

    wav_files = [file_name for file_name in files if file_name.endswith('.wav')]
    if len(wav_files) > 0 and AUDIO_DIR is None:
        AUDIO_DIR = root

print('Dataset CSV:  ', CSV_PATH)
print('Audio folder: ', AUDIO_DIR)
print('Output folder:', OUTPUT_DIR)
print('Audio files:  ', len([file_name for file_name in os.listdir(AUDIO_DIR) if file_name.endswith('.wav')]))
print(f'Notebook start time: {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(NOTEBOOK_START_TIME))}')


In [ ]:
# Step 2: Spectrogram and model hyperparameters

VARIANT_CONFIGS = {
    "short": {
        "segment_frames": 41,
        "segment_hop_frames": 20,   # ~50% overlap
        "num_epochs": 300,
        "learning_rate": 0.002,
    },
    "long": {
        "segment_frames": 101,
        "segment_hop_frames": 10,   # ~90% overlap
        "num_epochs": 150,
        "learning_rate": 0.01,
    },
}

# Shared spectrogram setup
SAMPLE_RATE = 22050
N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 60

# Shared model setup
NUM_CLASSES = 50
CONV1_FILTERS = 80
CONV2_FILTERS = 80
FC_UNITS = 5000
DROPOUT_RATE = 0.5

# Shared training setup
BATCH_SIZE = 1000
MOMENTUM = 0.9
WEIGHT_DECAY = 0.001

GRAD_CLIP_MAX_NORM = 5.0          
AUGMENT_TRAINING_DATA = True
NUM_AUG_VARIANTS = 4
MAX_TIME_DELAY_SEC = 0.5


In [ ]:
# Step 3: Load and explore the ESC-50 metadata

df = pd.read_csv(CSV_PATH)

display(df.head())

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [ ]:
# Step 4: Feature extraction — log-mel spectrogram + delta

def extract_features(audio_path, sr=SAMPLE_RATE, n_fft=N_FFT,
                     hop_length=HOP_LENGTH, n_mels=N_MELS):
  
    y, _ = librosa.load(audio_path, sr=sr)

    # normalize waveform
    y = librosa.util.normalize(y)

    mel_spec = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0
    )

    log_mel = librosa.power_to_db(mel_spec)
    delta = librosa.feature.delta(log_mel)

    features = np.stack([log_mel, delta], axis=0).astype(np.float32)
    return features


# Quick test on one file
test_file = os.path.join(AUDIO_DIR, df.iloc[0]['filename'])
test_feat = extract_features(test_file)

print('Feature shape for one clip:', test_feat.shape)
print('  Channel 0 (log-mel): min={:.2f}, max={:.2f}'.format(
    test_feat[0].min(), test_feat[0].max()))
print('  Channel 1 (delta):   min={:.2f}, max={:.2f}'.format(
    test_feat[1].min(), test_feat[1].max()))

Feature shape for one clip: (2, 60, 216)
  Channel 0 (log-mel): min=-62.73, max=17.27
  Channel 1 (delta):   min=-8.24, max=12.72


In [ ]:
# Step 5: Random time delays on the waveform

def extract_features_with_augmentation(
    audio_path,
    augment,
    num_variants=NUM_AUG_VARIANTS,
    max_delay_sec=MAX_TIME_DELAY_SEC,
    sr=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
):
    y, _ = librosa.load(audio_path, sr=sr)
    y = librosa.util.normalize(y)

    if augment:
        max_shift = int(round(max_delay_sec * sr))
        waveforms = [y]
        for _ in range(num_variants - 1):
            shift = np.random.randint(-max_shift, max_shift + 1)
            waveforms.append(np.roll(y, shift))
    else:
        waveforms = [y]

    features_list = []
    for waveform in waveforms:
        mel_spec = librosa.feature.melspectrogram(
            y=waveform, sr=sr, n_fft=n_fft,
            hop_length=hop_length, n_mels=n_mels, power=2.0,
        )
        log_mel = librosa.power_to_db(mel_spec)
        delta = librosa.feature.delta(log_mel)
        features_list.append(np.stack([log_mel, delta], axis=0).astype(np.float32))

    return features_list


print('Augmentation helper defined: extract_features_with_augmentation()')


In [ ]:
# Step 6: Segment extraction from full spectrograms

def extract_segments(features, segment_frames, segment_hop_frames):

    n_frames = features.shape[2]
    segments = []

    for start in range(0, n_frames - segment_frames + 1, segment_hop_frames):
        segment = features[:, :, start:start + segment_frames]

        segment_energy = np.mean(np.abs(segment[0]))
        if segment_energy > 1e-6:
            segments.append(segment.astype(np.float32))

    return segments


# Quick test on one clip for both variants
for variant_name, config in VARIANT_CONFIGS.items():
    test_segments = extract_segments(
        test_feat,
        segment_frames=config["segment_frames"],
        segment_hop_frames=config["segment_hop_frames"]
    )

    print(f"{variant_name} variant")
    print(f"  Full spectrogram frames: {test_feat.shape[2]}")
    print(f"  Number of segments:      {len(test_segments)}")
    if len(test_segments) > 0:
        print(f"  Segment shape:           {test_segments[0].shape}")
    print()

short variant
  Full spectrogram frames: 216
  Number of segments:      9
  Segment shape:           (2, 60, 41)

long variant
  Full spectrogram frames: 216
  Number of segments:      12
  Segment shape:           (2, 60, 101)



In [ ]:
# Step 7: Build segment samples and wrap them in a Dataset

def build_segment_samples(dataframe, audio_dir, segment_frames, segment_hop_frames,
                          augment=False):
    samples = []

    for idx in range(len(dataframe)):
        row = dataframe.iloc[idx]
        audio_path = os.path.join(audio_dir, row['filename'])
        label = int(row['target'])
        filename = row['filename']

        feature_variants = extract_features_with_augmentation(
            audio_path,
            augment=augment,
            num_variants=NUM_AUG_VARIANTS,
        )

        for features in feature_variants:
            segments = extract_segments(
                features,
                segment_frames=segment_frames,
                segment_hop_frames=segment_hop_frames,
            )
            for segment in segments:
                samples.append((segment, label, filename))

        if (idx + 1) % 200 == 0:
            print(f'  Processed {idx + 1}/{len(dataframe)} clips...')

    multiplier = NUM_AUG_VARIANTS if augment else 1
    print(f'  Total segments: {len(samples)} from {len(dataframe)} clips '
          f'(augment={augment}, variants={multiplier})')
    return samples


class ESC50Dataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        segment, label, filename = self.samples[idx]
        segment_tensor = torch.from_numpy(segment).float()
        return segment_tensor, label, filename


In [ ]:
# Step 8: Define the Piczak CNN model

class PiczakCNN(nn.Module):
    def __init__(self, segment_frames, num_classes=NUM_CLASSES, dropout=DROPOUT_RATE):
        super().__init__()

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        # Convolutional layers
        self.conv1 = nn.Conv2d(
            in_channels=2,
            out_channels=CONV1_FILTERS,
            kernel_size=(57, 6),
            stride=1,
            padding=0,
        )
        self.pool1 = nn.MaxPool2d(kernel_size=(4, 3), stride=(1, 3))

        self.conv2 = nn.Conv2d(
            in_channels=CONV1_FILTERS,
            out_channels=CONV2_FILTERS,
            kernel_size=(1, 3),
            stride=1,
            padding=0,
        )
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))

        # Flattened size depends on segment length
        self.flat_size = self._get_flat_size(segment_frames)

        # Fully connected layers
        self.fc1 = nn.Linear(self.flat_size, FC_UNITS)
        self.fc2 = nn.Linear(FC_UNITS, FC_UNITS)
        self.out = nn.Linear(FC_UNITS, num_classes)

        self._init_weights()

    def _get_flat_size(self, segment_frames):
        with torch.no_grad():
            x = torch.zeros(1, 2, N_MELS, segment_frames)
            x = self.relu(self.conv1(x))
            x = self.dropout(x)
            x = self.pool1(x)

            x = self.relu(self.conv2(x))
            x = self.pool2(x)

            return x.view(1, -1).shape[1]

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.dropout(x)
        x = self.pool1(x)

        x = self.relu(self.conv2(x))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.out(x)

        return x


# Quick verification for both variants
for variant_name, config in VARIANT_CONFIGS.items():
    model_test = PiczakCNN(segment_frames=config["segment_frames"]).to(device)

    dummy = torch.randn(1, 2, N_MELS, config["segment_frames"]).to(device)
    dummy_out = model_test(dummy)

    total_params = sum(p.numel() for p in model_test.parameters())
    trainable_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)

    del model_test, dummy, dummy_out


In [ ]:
# Step 9: Build fold-specific datasets and dataloaders

def _compute_channel_stats(samples):
    arr = np.stack([s[0] for s in samples], axis=0)         # (N, 2, 60, T)
    mean = arr.mean(axis=(0, 2, 3), keepdims=True)[0]       # (2, 1, 1)
    std  = arr.std(axis=(0, 2, 3), keepdims=True)[0] + 1e-6
    return mean.astype(np.float32), std.astype(np.float32)


def _apply_channel_stats(samples, mean, std):
    return [((seg - mean) / std, label, fname) for seg, label, fname in samples]


def make_fold_dataloaders(dataframe, audio_dir, test_fold, variant_name):
    config = VARIANT_CONFIGS[variant_name]

    train_df = dataframe[dataframe["fold"] != test_fold].reset_index(drop=True)
    test_df  = dataframe[dataframe["fold"] == test_fold].reset_index(drop=True)

    train_samples = build_segment_samples(
        train_df,
        audio_dir,
        segment_frames=config["segment_frames"],
        segment_hop_frames=config["segment_hop_frames"],
        augment=AUGMENT_TRAINING_DATA,
    )
    test_samples = build_segment_samples(
        test_df,
        audio_dir,
        segment_frames=config["segment_frames"],
        segment_hop_frames=config["segment_hop_frames"],
        augment=False,
    )

    channel_mean, channel_std = _compute_channel_stats(train_samples)

    train_samples = _apply_channel_stats(train_samples, channel_mean, channel_std)
    test_samples  = _apply_channel_stats(test_samples,  channel_mean, channel_std)

    train_dataset = ESC50Dataset(train_samples)
    test_dataset  = ESC50Dataset(test_samples)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

    return train_df, test_df, train_samples, test_samples, train_loader, test_loader


In [ ]:
# Step 10: Set up training helper

def _aggregate_clip_probs(probs, labels_cpu, fnames, store):
    for fn, prob, lab in zip(fnames, probs, labels_cpu):
        if fn not in store["sum"]:
            store["sum"][fn] = np.zeros(prob.shape[0], dtype=np.float64)
            store["label"][fn] = int(lab)
        store["sum"][fn] += prob


def _clip_accuracy(store):
    if not store["sum"]:
        return 0.0
    correct = 0
    for fn, prob_sum in store["sum"].items():
        if int(np.argmax(prob_sum)) == store["label"][fn]:
            correct += 1
    return correct / len(store["sum"])


def run_one_epoch(model, data_loader, loss_function, optimizer=None,
                  grad_clip_max_norm=None, nan_watchdog=False):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0
    clip_store = None if is_training else {"sum": {}, "label": {}}
    nan_skipped_batches = 0

    for batch_idx, (segments, labels, filenames) in enumerate(data_loader):
        segments = segments.to(device)
        labels = labels.to(device)

        if is_training:
            optimizer.zero_grad()
            logits = model(segments)
            loss = loss_function(logits, labels)

            # NaN watchdog — halve LR, skip step, do not poison weights.
            if nan_watchdog and not torch.isfinite(loss):
                nan_skipped_batches += 1
                for pg in optimizer.param_groups:
                    pg['lr'] *= 0.5
                print(
                    f'  [WARN] non-finite loss in batch {batch_idx}; '
                    f'halving LR -> {optimizer.param_groups[0]["lr"]:.2e}, '
                    f'skipping this batch'
                )
                optimizer.zero_grad()
                continue

            loss.backward()

            if grad_clip_max_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=grad_clip_max_norm
                )

            optimizer.step()
        else:
            with torch.no_grad():
                logits = model(segments)
                loss = loss_function(logits, labels)

        predictions = torch.argmax(logits, dim=1)

        batch_size_actual = labels.size(0)
        total_loss += loss.item() * batch_size_actual
        total_correct += (predictions == labels).sum().item()
        total_count += batch_size_actual

        if clip_store is not None:
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            _aggregate_clip_probs(probs, labels.cpu().numpy(), filenames, clip_store)

    if total_count == 0:
        return float('nan'), 0.0, None

    average_loss = total_loss / total_count
    average_accuracy = total_correct / total_count
    clip_accuracy = _clip_accuracy(clip_store) if clip_store is not None else None

    if is_training and nan_skipped_batches > 0:

    return average_loss, average_accuracy, clip_accuracy


def train_one_fold(dataframe, audio_dir, test_fold, variant_name):
    config = VARIANT_CONFIGS[variant_name]

    train_df, test_df, train_samples, test_samples, train_loader, test_loader = make_fold_dataloaders(
        dataframe,
        audio_dir,
        test_fold=test_fold,
        variant_name=variant_name,
    )

    model = PiczakCNN(
        segment_frames=config["segment_frames"],
        num_classes=NUM_CLASSES,
        dropout=DROPOUT_RATE,
    ).to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(),
        lr=config["learning_rate"],
        momentum=MOMENTUM,
        nesterov=True,
        weight_decay=WEIGHT_DECAY,
    )

    history = {
        "train_loss":    [],
        "train_acc":     [],
        "test_loss":     [],
        "test_acc":      [],   # segment-level
        "test_clip_acc": [],   # clip-level (probability voting; paper-comparable)
    }
    best_test_acc      = 0.0
    best_epoch         = 0
    best_clip_test_acc = 0.0
    best_clip_epoch    = 0

    for epoch_num in range(config["num_epochs"]):
        train_loss, train_acc, _ = run_one_epoch(
            model, train_loader, loss_function,
            optimizer=optimizer,
            grad_clip_max_norm=GRAD_CLIP_MAX_NORM,
            nan_watchdog=True,
        )

        test_loss, test_acc, test_clip_acc = run_one_epoch(
            model, test_loader, loss_function, optimizer=None,
        )

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch    = epoch_num + 1
        if test_clip_acc > best_clip_test_acc:
            best_clip_test_acc = test_clip_acc
            best_clip_epoch    = epoch_num + 1

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        history["test_clip_acc"].append(test_clip_acc)

        if (epoch_num + 1) % 10 == 0 or epoch_num == 0:
            print(
                f'Epoch {epoch_num + 1:03d}/{config["num_epochs"]:03d} | '
                f'train loss {train_loss:.4f}, acc {train_acc:.4f} | '
                f'test loss {test_loss:.4f} | '
                f'seg {test_acc:.4f} | clip {test_clip_acc:.4f} | '
                f'best clip {best_clip_test_acc:.4f} (ep {best_clip_epoch})'
            )

    return model, history


In [ ]:
# Step 11: Set up plot curves

def save_training_curves(history, variant_name, fold_number, output_dir):
    epoch_list = np.arange(1, len(history["train_loss"]) + 1)

    fig, axis_list = plt.subplots(1, 2, figsize=(12, 5))

    # Loss
    axis_list[0].plot(epoch_list, history["train_loss"], label="Train")
    axis_list[0].plot(epoch_list, history["test_loss"], label="Test")
    axis_list[0].set_title(f"{variant_name} fold {fold_number}: Loss")
    axis_list[0].set_xlabel("Epoch")
    axis_list[0].set_ylabel("Cross-Entropy Loss")
    axis_list[0].grid(True)
    axis_list[0].legend()

    # Accuracy (segment-level + clip-level probability voting)
    axis_list[1].plot(epoch_list, history["train_acc"], label="Train (segment)")
    axis_list[1].plot(epoch_list, history["test_acc"], label="Test (segment)")
    if "test_clip_acc" in history:
        axis_list[1].plot(
            epoch_list,
            history["test_clip_acc"],
            label="Test (clip, prob-vote)",
            linewidth=2,
        )
    axis_list[1].set_title(f"{variant_name} fold {fold_number}: Accuracy")
    axis_list[1].set_xlabel("Epoch")
    axis_list[1].set_ylabel("Accuracy")
    axis_list[1].grid(True)
    axis_list[1].legend()

    fig.tight_layout()

    plot_path = os.path.join(output_dir, f"{variant_name}_fold_{fold_number}_curves.png")
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)

    print(f"Training curves saved to: {plot_path}")

In [ ]:
# Step 12: Run all variants across all 5 folds

SAVE_PLOTS = True

all_results = []
history_records = []
all_histories = {}
variant_totals = {}                  # variant_name -> total seconds
training_loop_start = time.time()

for variant_name in VARIANT_CONFIGS:
    variant_start = time.time()

    for test_fold in range(1, 6):
        print('\n' + '=' * 60)
        print(f'Running {variant_name} variant — fold {test_fold}')
        print('=' * 60)

        fold_start = time.time()
        model, history = train_one_fold(
            df,
            AUDIO_DIR,
            test_fold=test_fold,
            variant_name=variant_name,
        )
        fold_seconds = time.time() - fold_start

        all_histories[(variant_name, test_fold)] = history

        best_test_acc      = max(history['test_acc'])
        best_epoch         = int(np.argmax(history['test_acc'])) + 1
        best_clip_test_acc = max(history['test_clip_acc'])
        best_clip_epoch    = int(np.argmax(history['test_clip_acc'])) + 1

        all_results.append({
            'variant':              variant_name,
            'fold':                 test_fold,
            'final_train_loss':     history['train_loss'][-1],
            'final_train_acc':      history['train_acc'][-1],
            'final_test_loss':      history['test_loss'][-1],
            'final_test_acc':       history['test_acc'][-1],       # segment-level
            'final_clip_test_acc':  history['test_clip_acc'][-1],  # clip-level (paper-comparable)
            'best_test_acc':        best_test_acc,
            'best_epoch':           best_epoch,
            'best_clip_test_acc':   best_clip_test_acc,
            'best_clip_epoch':      best_clip_epoch,
            'fold_seconds':         fold_seconds,
        })

        if SAVE_PLOTS:
            save_training_curves(history, variant_name, test_fold, OUTPUT_DIR)

        for epoch_idx in range(len(history['train_loss'])):
            history_records.append({
                'variant':        variant_name,
                'fold':           test_fold,
                'epoch':          epoch_idx + 1,
                'train_loss':     history['train_loss'][epoch_idx],
                'train_acc':      history['train_acc'][epoch_idx],
                'test_loss':      history['test_loss'][epoch_idx],
                'test_acc':       history['test_acc'][epoch_idx],       # segment-level
                'test_clip_acc':  history['test_clip_acc'][epoch_idx],  # clip-level
            })

        cumulative = time.time() - NOTEBOOK_START_TIME
        print(
            f'✓ {variant_name} fold {test_fold} done in '
            f'{_format_elapsed(fold_seconds)} '
            f'(cumulative since notebook start: {_format_elapsed(cumulative)})'
        )

    variant_totals[variant_name] = time.time() - variant_start
    print(
        f'\n>>> {variant_name} variant total: '
        f'{_format_elapsed(variant_totals[variant_name])} '
        f'across 5 folds'
    )

training_loop_seconds = time.time() - training_loop_start

print('\n' + '=' * 60)
print('ALL VARIANTS AND FOLDS COMPLETE')
print(f'Training loop total: {_format_elapsed(training_loop_seconds)}')
for variant_name, secs in variant_totals.items():
    print(f'  {variant_name:>5}: {_format_elapsed(secs)}')
print('=' * 60)


In [ ]:
# Step 13: Summarize results

results_df = pd.DataFrame(all_results)

summary_df = (
    results_df
    .groupby('variant')[[
        'final_test_acc',        # segment-level
        'final_clip_test_acc',   # clip-level (paper-comparable)
        'best_test_acc',
        'best_clip_test_acc',
        'fold_seconds',
    ]]
    .agg(['mean', 'std'])
    .round(4)
)


# Wall-clock elapsed since the very first cell ran.
total_elapsed_seconds = time.time() - NOTEBOOK_START_TIME
print('\nTiming:')
print(f'  Total notebook wall-clock time: {_format_elapsed(total_elapsed_seconds)} '
      f'({total_elapsed_seconds:.1f} s)')
for variant_name, secs in variant_totals.items():
    print(f'  {variant_name:>5} variant total (5 folds): {_format_elapsed(secs)}')

results_path = os.path.join(OUTPUT_DIR, 'baseline_results_by_fold.csv')
results_df.to_csv(results_path, index=False)

print(f'\nResults saved to: {results_path}')


In [ ]:
# Step 13: Save to CSV

def save_history_csv(history_records, output_dir):
    history_df = pd.DataFrame(history_records)

    history_path = os.path.join(output_dir, 'baseline_epoch_history.csv')
    history_df.to_csv(history_path, index=False)

    return history_df

In [ ]:
# Step 14: Save to CSV

def save_history_csv(history_records, output_dir):
    history_df = pd.DataFrame(history_records)

    history_path = os.path.join(output_dir, 'baseline_epoch_history.csv')
    history_df.to_csv(history_path, index=False)

    return history_df

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


results_df = pd.DataFrame(all_results)
history_df = pd.DataFrame(history_records)

required_history_columns = {
    "variant",
    "fold",
    "epoch",
    "train_loss",
    "test_loss",
    "train_acc",
    "test_acc",
    "test_clip_acc",
}
missing_history_columns = sorted(required_history_columns - set(history_df.columns))
if missing_history_columns:
    raise ValueError(
        f"history_records is missing {missing_history_columns}. "
        "Run the voting-enabled baseline notebook before this summary cell."
    )

required_result_columns = {
    "variant",
    "fold",
    "final_test_acc",
    "final_clip_test_acc",
    "best_clip_test_acc",
    "best_clip_epoch",
}
missing_result_columns = sorted(required_result_columns - set(results_df.columns))
if missing_result_columns:
    raise ValueError(
        f"all_results is missing {missing_result_columns}. "
        "Run the voting-enabled baseline notebook before this summary cell."
    )

PICZAK_SP_CLIP = 0.60
PICZAK_LP_CLIP = 0.645

variant_order = [variant for variant in ["short", "long"] if variant in history_df["variant"].unique()]
variant_labels = {
    "short": "Short / SP-style",
    "long": "Long / LP-style",
}
variant_colors = {
    "short": "#17677D",
    "long": "#B65D3A",
}
segment_colors = {
    "short": "#8DBDD0",
    "long": "#D6A33B",
}


def _mean_std_by_epoch(dataframe, variant, columns):
    return (
        dataframe[dataframe["variant"] == variant]
        .groupby("epoch")[columns]
        .agg(["mean", "std"])
        .reset_index()
    )


def _plot_mean_band(ax, x, mean, std, color, label, linewidth=2.6, linestyle="-", alpha=0.14):
    ax.plot(x, mean, color=color, linewidth=linewidth, linestyle=linestyle, label=label)
    if std is not None and np.isfinite(std).any():
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=alpha, linewidth=0)


summary_df = (
    results_df
    .groupby("variant")
    .agg(
        final_segment_mean=("final_test_acc", "mean"),
        final_segment_std=("final_test_acc", "std"),
        final_clip_mean=("final_clip_test_acc", "mean"),
        final_clip_std=("final_clip_test_acc", "std"),
        best_clip_mean=("best_clip_test_acc", "mean"),
        best_clip_std=("best_clip_test_acc", "std"),
        mean_best_epoch=("best_clip_epoch", "mean"),
    )
    .reindex(variant_order)
)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "legend.fontsize": 9,
})

fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
loss_ax, acc_ax = axes

for variant in variant_order:
    color = variant_colors[variant]
    grouped = _mean_std_by_epoch(history_df, variant, ["train_loss", "test_loss"])
    epochs = grouped["epoch"].to_numpy()

    train_loss_mean = grouped[("train_loss", "mean")].to_numpy()
    train_loss_std = grouped[("train_loss", "std")].fillna(0).to_numpy()
    test_loss_mean = grouped[("test_loss", "mean")].to_numpy()
    test_loss_std = grouped[("test_loss", "std")].fillna(0).to_numpy()

    _plot_mean_band(
        loss_ax,
        epochs,
        train_loss_mean,
        train_loss_std,
        color,
        f"{variant_labels[variant]} train loss",
        linewidth=2.4,
        linestyle="-",
        alpha=0.10,
    )
    _plot_mean_band(
        loss_ax,
        epochs,
        test_loss_mean,
        test_loss_std,
        color,
        f"{variant_labels[variant]} test loss",
        linewidth=2.4,
        linestyle="--",
        alpha=0.10,
    )

loss_ax.set_title("Training and Test Loss")
loss_ax.set_xlabel("Epoch")
loss_ax.set_ylabel("Cross-entropy loss")
loss_ax.grid(True, alpha=0.28)
loss_ax.legend(loc="upper right", frameon=True, framealpha=0.85)


for variant in variant_order:
    color = variant_colors[variant]
    faint_color = segment_colors[variant]
    variant_history = history_df[history_df["variant"] == variant]

    # Fold-level clip-vote curves provide the familiar "run output" texture.
    for fold, fold_df in variant_history.groupby("fold"):
        acc_ax.plot(
            fold_df["epoch"].to_numpy(),
            fold_df["test_clip_acc"].to_numpy() * 100,
            color=color,
            alpha=0.20,
            linewidth=1.1,
        )

    grouped = _mean_std_by_epoch(history_df, variant, ["test_acc", "test_clip_acc"])
    epochs = grouped["epoch"].to_numpy()

    segment_mean = grouped[("test_acc", "mean")].to_numpy() * 100
    clip_mean = grouped[("test_clip_acc", "mean")].to_numpy() * 100
    clip_std = grouped[("test_clip_acc", "std")].fillna(0).to_numpy() * 100

    # Segment accuracy is shown quietly because it is diagnostic, not Piczak-comparable.
    acc_ax.plot(
        epochs,
        segment_mean,
        color=faint_color,
        linewidth=1.8,
        linestyle=":",
        label=f"{variant_labels[variant]} test segment",
    )
    _plot_mean_band(
        acc_ax,
        epochs,
        clip_mean,
        clip_std,
        color,
        f"{variant_labels[variant]} test clip vote",
        linewidth=3.0,
        linestyle="-",
        alpha=0.12,
    )

acc_ax.axhline(PICZAK_SP_CLIP * 100, color="#6B7176", linestyle="--", linewidth=1.4, label="Piczak SP approx. 60%")
acc_ax.axhline(PICZAK_LP_CLIP * 100, color="#1F2528", linestyle="--", linewidth=1.7, label="Piczak LP 64.5%")

for variant in variant_order:
    if variant not in summary_df.index:
        continue
    final_clip = summary_df.loc[variant, "final_clip_mean"] * 100
    best_clip = summary_df.loc[variant, "best_clip_mean"] * 100
    if np.isfinite(final_clip):
        acc_ax.scatter(
            [history_df[history_df["variant"] == variant]["epoch"].max()],
            [final_clip],
            s=70,
            color=variant_colors[variant],
            edgecolor="white",
            linewidth=1.0,
            zorder=4,
        )
        acc_ax.annotate(
            f"{variant.split()[0].capitalize()} final {final_clip:.1f}%\nBest {best_clip:.1f}%",
            xy=(history_df[history_df["variant"] == variant]["epoch"].max(), final_clip),
            xytext=(-70, 18 if variant == "long" else -42),
            textcoords="offset points",
            fontsize=9,
            color=variant_colors[variant],
            arrowprops=dict(arrowstyle="-", color=variant_colors[variant], lw=1),
        )

acc_ax.set_title("Model Accuracy")
acc_ax.set_xlabel("Epoch")
acc_ax.set_ylabel("Accuracy (%)")
acc_ax.set_ylim(0, 100)
acc_ax.grid(True, alpha=0.28)
acc_ax.legend(loc="lower right", frameon=True, framealpha=0.88)

fig.suptitle(
    "Piczak CNN baseline",
    fontsize=20,
    fontweight="bold",
)

summary_png_path = os.path.join(OUTPUT_DIR, "baseline_piczak_comparable_curves.png")
summary_csv_path = os.path.join(OUTPUT_DIR, "baseline_piczak_comparable_summary.csv")
summary_df.to_csv(summary_csv_path)

fig.savefig(summary_png_path, dpi=220, bbox_inches="tight")
plt.show()

print("Saved Piczak-comparable curve figure to:", summary_png_path)
print("Saved Piczak-comparable summary table to:", summary_csv_path)
display((summary_df * 100).round(2))